# Clustering & Anomaly Detection — Healthcare Utilization

Demonstrates **clustering** (patient/encounter segments) and **anomaly detection** (outlier claims/encounters) on the healthcare mock dataset. Aligns with JD: ML for clustering, classification, regression, anomaly detection.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
import matplotlib.pyplot as plt

# Repo root: from notebooks/python go up twice; from repo root use data/raw
if (Path.cwd() / "data" / "raw").exists():
    DATA_DIR = Path.cwd() / "data" / "raw"
else:
    DATA_DIR = Path.cwd().parent.parent / "data" / "raw"

## 1. Load and build encounter-level features

In [ ]:
encounters = pd.read_csv(DATA_DIR / "encounters.csv", parse_dates=["admit_date", "discharge_date"])
patients = pd.read_csv(DATA_DIR / "patients.csv")
diagnoses = pd.read_csv(DATA_DIR / "diagnoses.csv")
claims = pd.read_csv(DATA_DIR / "claims.csv")

encounters["los_days"] = (
    (encounters["discharge_date"] - encounters["admit_date"]).dt.total_seconds() / 86400
).clip(lower=0)
encounters["admit_date_d"] = pd.to_datetime(encounters["admit_date"])
patients["date_of_birth"] = pd.to_datetime(patients["date_of_birth"])
encounters = encounters.merge(
    patients[["patient_id", "date_of_birth", "gender"]], on="patient_id", how="left"
)
encounters["age"] = (
    (encounters["admit_date_d"] - encounters["date_of_birth"]).dt.days / 365.25
).clip(0, 120)

diag_count = diagnoses.groupby("encounter_id").size().reset_index(name="diag_count")
encounters = encounters.merge(diag_count, on="encounter_id", how="left")
encounters["diag_count"] = encounters["diag_count"].fillna(0).astype(int)

encounters = encounters.merge(
    claims[["encounter_id", "total_charge", "total_paid"]], on="encounter_id", how="left"
)
encounters["total_paid"] = encounters["total_paid"].fillna(0)
encounters["total_charge"] = encounters["total_charge"].fillna(0)

## 2. Clustering — segment encounters by utilization and acuity

In [ ]:
feat_cols = ["los_days", "age", "diag_count", "total_paid"]
X = encounters[feat_cols].fillna(0)
X = X.replace([np.inf, -np.inf], 0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
encounters["cluster"] = kmeans.fit_predict(X_scaled)

cluster_summary = encounters.groupby("cluster")[feat_cols].agg(["mean", "count"]).round(2)
print("Cluster summaries (mean):")
print(cluster_summary)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
for c in sorted(encounters["cluster"].unique()):
    sub = encounters[encounters["cluster"] == c]
    ax.scatter(sub["los_days"], sub["total_paid"], alpha=0.3, s=10, label=f"Cluster {c}")
ax.set_xlabel("Length of stay (days)")
ax.set_ylabel("Total paid")
ax.set_title("Encounter clusters: LOS vs total paid")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Anomaly detection — flag outlier encounters

In [ ]:
iso = IsolationForest(contamination=0.05, random_state=42)
encounters["anomaly_score"] = iso.fit_predict(X_scaled)  # -1 = anomaly
encounters["is_anomaly"] = (encounters["anomaly_score"] == -1).astype(int)

n_anom = encounters["is_anomaly"].sum()
print(f"Anomalies flagged: {n_anom} ({100 * n_anom / len(encounters):.2f}%)")
print(encounters[encounters["is_anomaly"] == 1][feat_cols].describe().round(2))

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
norm = encounters[encounters["is_anomaly"] == 0]
anom = encounters[encounters["is_anomaly"] == 1]
ax.scatter(norm["los_days"], norm["total_paid"], alpha=0.2, s=8, c="steelblue", label="Normal")
ax.scatter(anom["los_days"], anom["total_paid"], alpha=0.7, s=20, c="coral", label="Anomaly")
ax.set_xlabel("Length of stay (days)")
ax.set_ylabel("Total paid")
ax.set_title("Anomaly detection: outlier encounters (Isolation Forest)")
ax.legend()
plt.tight_layout()
plt.show()

## Summary

- **Clustering**: KMeans on LOS, age, diagnosis count, and total paid segments encounters into utilization/acuity groups for reporting or targeting.
- **Anomaly detection**: Isolation Forest flags ~5% of encounters as outliers (e.g. unusual cost or LOS) for review.